# Reto 05 — SOLUCION

No liberar hasta cerrar el modulo.

In [ ]:
import pandas as pd

DS = "../../../_RECURSOS/datasets"

ventas    = pd.read_csv(f"{DS}/ventas.csv",    sep=";", decimal=",", encoding="utf-8")
clientes  = pd.read_csv(f"{DS}/clientes.csv",  sep=";", encoding="utf-8")
productos = pd.read_csv(f"{DS}/productos.csv", sep=";", encoding="utf-8")

print(ventas.shape, clientes.shape, productos.shape)

## 1 — Limpiar

In [ ]:
ventas = ventas.drop_duplicates().reset_index(drop=True)
ventas["canal"] = ventas["canal"].str.strip().str.title()
ventas["fecha"] = pd.to_datetime(ventas["fecha"], format="mixed", dayfirst=True)
ventas["importe"] = ventas["importe"].fillna(
    (ventas["cantidad"] * ventas["precio_unitario"]
     * (1 - ventas["descuento_pct"] / 100)).round(2)
)

assert len(ventas) == 482 and ventas["importe"].isna().sum() == 0
print("OK 1 —", ventas.shape)

## 2 — Diagnostico previo

Este paso existe porque `merge` **no avisa**. Si cruzas a ciegas con `inner`, pandas
borra la fila huerfana y te devuelve un DataFrame perfectamente saludable con menos plata
adentro. Nadie te va a decir nada hasta que el total no cuadre con contabilidad.

In [ ]:
huerfanas = ventas.loc[~ventas["id_cliente"].isin(clientes["id_cliente"])]
sin_ventas = clientes.loc[~clientes["id_cliente"].isin(ventas["id_cliente"])]
prod_ok = ventas["id_producto"].isin(productos["id_producto"]).all()

print("Ventas con cliente inexistente:", len(huerfanas))
print(huerfanas[["id_venta", "id_cliente", "importe"]].to_string(index=False))
print("\nClientes sin ninguna venta:", len(sin_ventas))
print(sin_ventas[["id_cliente", "nombre"]].to_string(index=False))
print("\nTodos los productos existen en el catalogo:", prod_ok)

**Hallazgos:**

1. **`C777` es una venta con un cliente que no existe en el maestro.** No es un problema de
   pandas, es un problema del sistema origen: o el cliente se dio de baja, o hubo un typo al
   registrar. La venta **ocurrio** y la plata entro. Si la borras, el reporte de ventas no
   cuadra con caja.
2. **`C099` es un cliente sin ninguna compra.** Para un reporte de ventas es ruido; para uno
   de "clientes activos vs inactivos" es justamente el dato interesante. Depende de la pregunta.

## 3 — `inner` vs `left`

In [ ]:
con_inner = ventas.merge(clientes, on="id_cliente", how="inner")
con_left  = ventas.merge(clientes, on="id_cliente", how="left")

print(f"inner: {len(con_inner)}  |  left: {len(con_left)}")
print("\nLa fila que pierde el inner:")
print(con_left.loc[con_left["nombre"].isna(), ["id_venta", "id_cliente", "importe"]].to_string(index=False))

assert len(con_inner) == 481 and len(con_left) == 482
print("\nOK 3")

**Decision: `left`.** El DataFrame de la izquierda son los hechos (las ventas ocurrieron);
el de la derecha es contexto que puede faltar. Se conserva la venta con `nombre` en `NaN`
y se rellena con `"Cliente no identificado"`, para que la fila sea visible en el reporte
en vez de desaparecer.

Regla general: **cuando la tabla izquierda son hechos y la derecha es un maestro, va `left`.**
El `inner` se usa cuando de verdad quieres solo la interseccion, y eso es una decision
que se toma, no un default.

## 4 — Tabla maestra

`validate="m:1"` es la red de seguridad: le dice a pandas "muchas ventas, un cliente".
Si el maestro tuviera un `id_cliente` duplicado, el merge multiplicaria filas y el total se
inflaria. Con `validate`, pandas lanza `MergeError` en vez de darte un numero mas grande y falso.

In [ ]:
maestra = (
    ventas
    .merge(clientes, on="id_cliente", how="left",
           suffixes=("", "_cliente"), validate="m:1")
    .merge(productos, on="id_producto", how="left",
           suffixes=("", "_catalogo"), validate="m:1")
)

maestra["nombre"] = maestra["nombre"].fillna("Cliente no identificado")
maestra["segmento"] = maestra["segmento"].fillna("Desconocido")

assert len(maestra) == 482
print("OK 4 —", maestra.shape)
print(maestra.columns.tolist())

> **Ojo con `ciudad`:** existe en `ventas` (donde se hizo la venta) y en `clientes`
> (donde vive el cliente). Con `suffixes=("", "_cliente")` la de ventas conserva el nombre
> y la del cliente queda como `ciudad_cliente`. Son dos cosas distintas y confundirlas
> cambia el resultado del analisis geografico.

## 5 — Agregaciones

In [ ]:
# 5.1 — una sola llamada a agg, no tres groupby
por_segmento = maestra.groupby("segmento").agg(
    importe_total=("importe", "sum"),
    n_ventas=("id_venta", "count"),
    ticket_promedio=("importe", "mean"),
).round(2).sort_values("importe_total", ascending=False)

print(por_segmento.to_string())

In [ ]:
# 5.2
por_cat_canal = (maestra.groupby(["categoria", "canal"])["importe"]
                 .sum().round(2).unstack(fill_value=0))
print(por_cat_canal.to_string())

In [ ]:
# 5.3 — mes top por ciudad
maestra["mes"] = maestra["fecha"].dt.to_period("M")
por_ciudad_mes = maestra.groupby(["ciudad", "mes"])["importe"].sum()
mes_top = por_ciudad_mes.groupby(level="ciudad").idxmax().apply(lambda t: t[1])
valor_top = por_ciudad_mes.groupby(level="ciudad").max().round(2)

print(pd.DataFrame({"mes_top": mes_top.astype(str), "importe": valor_top}).to_string())

## 6 — Pivot

In [ ]:
maestra["mes_str"] = maestra["mes"].astype(str)

tabla = maestra.pivot_table(
    index="ciudad", columns="mes_str",
    values="importe", aggfunc="sum", fill_value=0,
    margins=True, margins_name="TOTAL",
).round(0)

print(tabla.to_string())


## 7 — Exportar

In [ ]:
salida = maestra.drop(columns=["mes", "mes_str"])
salida.to_csv("ventas_maestra.csv", sep=";", decimal=",", index=False, encoding="utf-8")

import os
assert os.path.exists("ventas_maestra.csv")
print("OK 7 —", len(salida), "filas")

## 8 — API

`timeout` siempre. Sin el, si la API no responde el notebook se queda colgado para siempre
y el alumno cree que Jupyter se rompio.

In [ ]:
import requests

try:
    r = requests.get("https://open.er-api.com/v6/latest/USD", timeout=10)
    r.raise_for_status()
    tc = r.json()["rates"]["PEN"]
    print(f"Tipo de cambio USD/PEN: {tc}")
    print(f"Facturacion en USD: $ {maestra['importe'].sum() / tc:,.2f}")
except (requests.RequestException, KeyError) as e:
    print(f"No se pudo consultar la API ({type(e).__name__}). Se usa el tipo de cambio fijo.")
    tc = 3.75
    print(f"Facturacion en USD (tc fijo {tc}): $ {maestra['importe'].sum() / tc:,.2f}")

---
### Errores tipicos al corregir

- **Cruzar sin diagnosticar.** Es el error central del modulo. `inner` a ciegas y adios a una venta.
- **`merge` sin `how=`.** El default es `inner`, y un default no es una decision.
- **No usar `validate=`.** Un duplicado en el maestro infla el total y el merge no se queja.
- **Ignorar los sufijos** y quedarse con `ciudad_x` / `ciudad_y` sin saber cual es cual.
- **Tres `groupby` separados** en el 5.1 en vez de un `agg` con tuplas nombradas.
- **Agrupar el mes por `dt.month`** y mezclar meses de anos distintos (aqui no se nota porque solo hay 2025, pero es una bomba de tiempo).
- **`requests.get` sin `timeout`.**